# Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from fontTools.subset import subset
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
import warnings

from transformers.initialization import default_flax_embed_init_

warnings.filterwarnings('ignore', category=UserWarning, message='.*unknown categories.*')

In [2]:
# Defining a data audit function that will help quickly evaluate key characteristics of the dataset
def data_audit(df: pd.DataFrame, n_unique_preview: int = 8) -> pd.DataFrame:
    summary = []
    for col in df.columns:
        s = df[col]

        # Calculate min/max only for numeric columns
        if pd.api.types.is_numeric_dtype(s):
            min_val = float(s.min())
            median_val = float(s.median())
            max_val = float(s.max())
        else:
            min_val = None
            median_val = None
            max_val = None

        summary.append({
            'column': col,
            'dtype': str(s.dtype),
            'n_missing': int(s.isna().sum()),
            'pct_missing': float(s.isna().mean()),
            'n_unique': int(s.nunique(dropna=True)),
            'min': min_val,
            'median': median_val,
            'max': max_val,
            "example_values": ", ".join(map(str, s.dropna().unique()[:n_unique_preview])),
        })

    out = pd.DataFrame(summary).sort_values(
        ["pct_missing", "n_unique"], ascending=[False, True]
    )
    return out

In [3]:
df = pd.read_csv('../data/processed_data/processed_data.csv')

We'll do a quick audit to see if everything imported properly.

In [4]:
data_audit(df)

,column,dtype,n_missing,pct_missing,n_unique,min,median,max,example_values
9,age,float64,3394,0.532559,32,0.000,7.000,37.000,"9.0, 6.0, 4.0, 8.0, 5.0, 10.0, 7.0, 3.0"
6,t_length,float64,1198,0.187981,773,41.000,521.000,1168.000,"439.0, 425.0, 418.0, 409.0, 398.0, 405.0, 394...."
5,f_length,float64,1083,0.169936,751,61.000,485.000,1105.000,"383.0, 370.0, 374.0, 355.0, 347.0, 354.0, 348...."
7,weight,float64,137,0.021497,2327,0.001,1.150,10.700,"0.61, 0.647, 0.71, 0.591, 0.6, 0.558, 0.621, 0..."
10,hg,float64,6,0.000941,1095,0.001,0.229,2.552,"0.136, 0.104, 0.057, 0.082, 0.143, 0.101, 0.07..."
4,sex,str,0,0.000000,3,NaN,NaN,NaN,"Female, Male, Unknown"
8,maturity,str,0,0.000000,4,NaN,NaN,NaN,"Mature, Immature, Unknown, Triploid"
2,w_type,str,0,0.000000,5,NaN,NaN,NaN,"Reservoir, Lake, River, Canal, Stormwater Pond"
11,year,int64,0,0.000000,21,1997.000,2013.000,2021.000,"2021, 2013, 2010, 2018, 2014, 2012, 2011, 2017"
3,common_name,str,0,0.000000,22,NaN,NaN,NaN,"Lake Whitefish, Northern Pike, Walleye, Mounta..."


Looks good except we need to convert `coll_date` back to a datetime dtype again.

In [5]:
df['coll_date'] = pd.to_datetime(df['coll_date'])
df['year'] = df['year'].astype(str)

In [6]:
hg_threshold = 0.2
df['safety_status'] = np.where(df['hg'] < hg_threshold, 'safe', 'unsafe')

We'll need to define imputers that will replace any NaN values before feeding the data into the model.

In [7]:
# Create imputer for numeric data (fill with median)
numerical_imputer = SimpleImputer(strategy='median')

# Create imputer for categorical data (fill with most frequent value)
categorical_imputer = SimpleImputer(strategy='most_frequent')

In [8]:
# Define features
categorical_features = ['w_name', 'w_type', 'common_name', 'sex', 'maturity', 'year']
numerical_features = ['f_length', 't_length', 'weight', 'age']

X = df[categorical_features + numerical_features]
y = df['safety_status']

In [9]:
preprocessor = ColumnTransformer([
    ('num_pipe', Pipeline([
        ('impute', numerical_imputer),
        ('scale', StandardScaler())
    ]), numerical_features),

    ('cat_pipe', Pipeline([
        ('impute', categorical_imputer),
        ('encode', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), categorical_features)
])

In [10]:
def transform_to_dataframe(preprocessor, X, numerical_features, categorical_features):
    # Fit and transform
    X_transformed = preprocessor.fit_transform(X)

    # Get feature names from the fitted transformer
    cat_encoder = preprocessor.named_transformers_['cat_pipe'].named_steps['encode']
    cat_feature_names = cat_encoder.get_feature_names_out(categorical_features)

    # Combine numerical and categorical feature names
    all_feature_names = list(numerical_features) + list(cat_feature_names)

    # Create DataFrame
    X_df = pd.DataFrame(X_transformed, columns=all_feature_names)

    return X_df

# Usage
X_processed = transform_to_dataframe(preprocessor, X, numerical_features, categorical_features)

In [11]:
X_processed.to_csv('../data/processed_data/features_processed/model_features.csv', index=False)
y.to_csv('../data/processed_data/features_processed/model_target.csv', index=False)

# Features for Minimal Model

In [12]:
categorical_features_mini = ['w_name']
numerical_features_mini = ['t_length', 'weight']

X_mini = df[categorical_features_mini + numerical_features_mini]
y_mini = df['safety_status']

In [13]:
preprocessor_mini = ColumnTransformer([
    ('num_pipe', Pipeline([
        ('impute', numerical_imputer),
        ('scale', StandardScaler())
    ]), numerical_features_mini),

    ('cat_pipe', Pipeline([
        ('impute', categorical_imputer),
        ('encode', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), categorical_features_mini)
])

In [14]:
X_mini_processed = transform_to_dataframe(preprocessor_mini, X_mini, numerical_features_mini, categorical_features_mini)

In [15]:
X_mini_processed.to_csv('../data/processed_data/features_processed/model_features_mini.csv', index=False)
y_mini.to_csv('../data/processed_data/features_processed/model_target_mini.csv', index=False)

# Minimal features with NaN values

In [16]:
categorical_features_bare = ['w_name']
numerical_features_bare = ['t_length', 'weight']

X_bare = df[categorical_features_bare + numerical_features_bare]
y_bare = df['safety_status']

In [17]:
preprocessor_no_impute = ColumnTransformer([
    ('num_pipe', Pipeline([
        ('scale', StandardScaler())
    ]), numerical_features_mini),

    ('cat_pipe', Pipeline([
        ('encode', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), categorical_features_mini)
])

In [18]:
X_bare_processed = transform_to_dataframe(preprocessor_no_impute, X_bare, numerical_features_bare, categorical_features_bare)

In [19]:
X_bare_processed.to_csv('../data/processed_data/features_processed/model_features_bare.csv', index=False)
y_bare.to_csv('../data/processed_data/features_processed/model_target_bare.csv', index=False)

# Complete model with NaN values

In [20]:
categorical_features_w_nan = ['w_name', 'w_type', 'common_name', 'sex', 'maturity', 'year']
numerical_features_w_nan  = ['f_length', 't_length', 'weight', 'age']

X_w_nan = df[categorical_features_w_nan + numerical_features_w_nan]
y_w_nan = df['safety_status']

In [21]:
preprocessor_no_impute = ColumnTransformer([
    ('num_pipe', Pipeline([
        ('scale', StandardScaler())
    ]), numerical_features_w_nan),

    ('cat_pipe', Pipeline([
        ('encode', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), categorical_features_w_nan)
])

In [22]:
X_w_nan_processed = transform_to_dataframe(preprocessor_no_impute, X_w_nan, numerical_features_w_nan, categorical_features_w_nan)

In [23]:
X_w_nan_processed.to_csv('../data/processed_data/features_processed/model_features_w_nan.csv', index=False)
y_w_nan.to_csv('../data/processed_data/features_processed/model_target_w_nan.csv', index=False)